## Prompt Templates & Message Structures - Practice Notebook (Google Colab)
This notebook corresponds to the **02-Prompt Templates & Message Structures** documentation page. Follow the concepts and run the code cells.

### 🛠️ Setup Environment & Load API Keys

In [ ]:
# Install core dependencies in Google Colab
!pip install -q langchain-core langchain-google-genai langchain-groq python-dotenv

In [ ]:
import os
import getpass

# Set environment variables dynamically in Colab
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Google API Key: ')

### 1. PromptTemplate (String-Based Prompts)

`PromptTemplate` is used to create simple, string-based prompts. It is ideal for basic text generation pipelines.

#### Example 1: Concept Explanation

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

# Define a template with a variable {concept}
template_str = "Explain the concept of {concept} in simple terms."
prompt_template = PromptTemplate.from_template(template_str)

# Fill in the variable
prompt = prompt_template.invoke({"concept": "machine learning"})
print("--- Formatted Prompt ---")
print(prompt.to_string())

# Initialize the chat model and invoke
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
response = llm.invoke(prompt)
print("\n--- LLM Response ---")
print(response.content)

#### Example 2: Automated Code Reviewer

In [ ]:
reviewer_template = PromptTemplate.from_template(
    "Review the following {language} code for security vulnerabilities and performance bottlenecks:\n\n```{language}\n{code}\n```"
)

code_to_review = """
def read_file(filename):
    import os
    os.system('cat ' + filename)
"""

formatted_prompt = reviewer_template.format(
    language="Python", 
    code=code_to_review
)
print("--- Formatted Prompt ---")
print(formatted_prompt)

response = llm.invoke(formatted_prompt)
print("\n--- LLM Response ---")
print(response.content)

#### Exercises for PromptTemplate

##### Exercise 1: Recipe Generator
Define a `PromptTemplate` that takes an `ingredients` list (e.g., "tomato, cheese, basil") and a `cuisine` type (e.g., "Italian"), and prompts the model to generate a recipe.

<details>
<summary><b>Click to view solution</b></summary>

```python
recipe_template = PromptTemplate.from_template(
    "Create a traditional {cuisine} recipe using these ingredients: {ingredients}."
)
prompt = recipe_template.invoke({"cuisine": "Italian", "ingredients": "tomato, cheese, basil"})
print(prompt.to_string())
```
</details>

In [ ]:
# Write your code here

##### Exercise 2: Technical Definition Writer
Define a `PromptTemplate` that takes a `term` and an `audience_level` (e.g., "5-year-old" or "PhD student") and generates a customized definition.

<details>
<summary><b>Click to view solution</b></summary>

```python
definition_template = PromptTemplate.from_template(
    "Define the term '{term}' in a way that a {audience_level} can easily understand."
)
prompt = definition_template.invoke({"term": "Quantum Computing", "audience_level": "5-year-old"})
print(prompt.to_string())
```
</details>

In [ ]:
# Write your code here

### 2. Message Types & Chat Structures

Chat models use lists of message objects rather than raw strings to maintain role-based boundaries.
* **`SystemMessage`**: Sets the behavior / rules.
* **`HumanMessage`**: Represents user input.
* **`AIMessage`**: Represents model responses.

#### Invoking ChatModels with Message Objects

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Construct dialogue list
messages = [
    SystemMessage(content="You are a strict Python security auditor. Review the user's code for safety issues."),
    HumanMessage(content="Here is my code:\n\nx = input('Enter command: ')\neval(x)")
]

response = llm.invoke(messages)
print("--- Auditor Response ---")
print(response.content)

### 3. ChatPromptTemplate (Message-Based Prompts)

`ChatPromptTemplate` structures conversation flows dynamically.

#### Example 1: Customer Service Ticket Auto-Classifier

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an automated support classifier. Categorize the ticket into one of: Hardware, Software, Billing."),
    ("human", "Ticket: {ticket_description}")
])

prompt = chat_template.format_messages(ticket_description="My screen keeps flickering.")
for msg in prompt:
    print(f"{msg.type.upper()}: {msg.content}")

response = llm.invoke(prompt)
print("\nResponse:", response.content)

#### Example 2: Geography Expert (Few-Shot Chat)

In [ ]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert that returns the colors present in a country's flag."),
    ("human", "France"),
    ("ai", "blue, white, red"),
    ("human", "{country}")
])

prompt = chat_template.invoke({"country": "Japan"})
response = llm.invoke(prompt)
print("Response:", response.content)

#### Exercises for ChatPromptTemplate

##### Exercise 1: History Guide Roleplay
Create a `ChatPromptTemplate` simulating a historical dialogue.
* System message: `"You are {historical_figure}, a historical figure. Answer in their character."`
* Human: `"What was your greatest achievement?"`
* AI: `"My greatest achievement was {achievement}."`
* Human: `"Why was {achievement} important?"`

<details>
<summary><b>Click to view solution</b></summary>

```python
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are {historical_figure}, a historical figure. Answer in their character."),
    ("human", "What was your greatest achievement?"),
    ("ai", "My greatest achievement was {achievement}."),
    ("human", "Why was {achievement} important?")
])
prompt = chat_template.invoke({"historical_figure": "Julius Caesar", "achievement": "crossing the Rubicon"})
for msg in prompt.to_messages():
    print(f"{msg.type.upper()}: {msg.content}")
```
</details>

In [ ]:
# Write your code here

##### Exercise 2: Code Translator
Create a `ChatPromptTemplate` representing a code translation engine.
* System message: `"You are an expert software engineer that translates source code from {source_lang} to {target_lang}."`
* Human: `"Translate this code:\n\n{code}"`

<details>
<summary><b>Click to view solution</b></summary>

```python
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert software engineer that translates source code from {source_lang} to {target_lang}."),
    ("human", "Translate this code:\n\n{code}")
])
prompt = chat_template.invoke({"source_lang": "Python", "target_lang": "JavaScript", "code": "print('Hello World')"})
for msg in prompt.to_messages():
    print(f"{msg.type.upper()}: {msg.content}")
```
</details>

In [ ]:
# Write your code here

### 4. Variable Passing Mechanisms

LangChain automatically handles single placeholder shortcuts or multi-variable dictionaries.

In [ ]:
# Single variable shortcut
single_prompt = ChatPromptTemplate.from_template("Explain {topic}")
print(single_prompt.invoke("Python").to_messages())

# Multi variable dictionary
multi_prompt = ChatPromptTemplate.from_template("Translate {text} to {lang}")
print(multi_prompt.invoke({"text": "Hello", "lang": "Spanish"}).to_messages())

### 5. Extracting Responses: `.content` vs `.text` vs Direct Output

In [ ]:
# 1. Use .content for ChatModel responses
response = llm.invoke("Hi")
print("ChatModel .content:", response.content)

# 2. Use .text for PromptValue formats returned by legacy/Few-Shot templates
# prompt_val.text returns the raw string